# 📊 MÓDULO 4: Evaluación de Modelos
## Métricas, Matriz de Confusión y Comparación Final

**Objetivo:** Evaluar ambos modelos y decidir cuál recomendar.

---

## 4.1 Configuración y Entrenamiento Rápido

In [1]:
import sys
import time

EN_COLAB = 'google.colab' in sys.modules
RUTA_DATOS = '/content/drive/MyDrive/ML_Vuelos/data/' if EN_COLAB else '../data/'

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Vuelos_Modulo4").config("spark.driver.memory", "4g").getOrCreate()

# Preparar datos
from pyspark.ml.feature import StringIndexer, VectorAssembler
df = spark.read.csv(RUTA_DATOS + "flights_2015_full.csv", header=True, inferSchema=True)

for col_name in ["AIRLINE", "ORIGIN", "DEST"]:
    indexer = StringIndexer(inputCol=col_name, outputCol=f"{col_name}_idx", handleInvalid="keep")
    df = indexer.fit(df).transform(df)

feature_cols = ["MONTH", "DAY_OF_WEEK", "DEP_HOUR", "DISTANCE", "AIRLINE_idx", "ORIGIN_idx", "DEST_idx"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_ml = assembler.transform(df).select("features", "DEP_DEL15")
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

#valanceo del modelo
retrasados = train_df.filter(train_df.DEP_DEL15 == 1)
a_tiempo = train_df.filter(train_df.DEP_DEL15 == 0)

# Calculamos la proporción
fraccion = retrasados.count() / a_tiempo.count()

# Muestreamos la clase mayoritaria
a_tiempo_sub = a_tiempo.sample(withReplacement=False, fraction=fraccion, seed=42)

# Unimos para crear el set de entrenamiento balanceado
train_df_balanced = retrasados.union(a_tiempo_sub)

# Entrenar modelos
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier

print("🌳 Entrenando Árbol...")
dt = DecisionTreeClassifier(labelCol="DEP_DEL15", featuresCol="features", maxDepth=8,minInstancesPerNode=100,maxBins=330)
modelo_arbol = dt.fit(train_df_balanced)

print("🌲 Entrenando Bosque...")
rf = RandomForestClassifier(labelCol="DEP_DEL15", featuresCol="features", numTrees=80, maxDepth=8, seed=42,maxBins=330)
modelo_bosque = rf.fit(train_df_balanced)

print("✅ Modelos listos")

🌳 Entrenando Árbol...
🌲 Entrenando Bosque...
✅ Modelos listos


## 4.2 Predicciones

In [2]:
# Hacer predicciones en test
pred_arbol = modelo_arbol.transform(test_df)
pred_bosque = modelo_bosque.transform(test_df)

print("📊 Predicciones del Árbol:")
pred_arbol.select("DEP_DEL15", "prediction", "probability").show(5)

📊 Predicciones del Árbol:
+---------+----------+--------------------+
|DEP_DEL15|prediction|         probability|
+---------+----------+--------------------+
|        0|       1.0|[0.36916150815981...|
|        0|       1.0|[0.36916150815981...|
|        0|       1.0|[0.36916150815981...|
|        1|       0.0|[0.58404558404558...|
|        0|       1.0|[0.36916150815981...|
+---------+----------+--------------------+
only showing top 5 rows



## 4.3 Métricas de Evaluación

In [3]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Evaluadores
eval_acc = MulticlassClassificationEvaluator(labelCol="DEP_DEL15", metricName="accuracy")
eval_f1 = MulticlassClassificationEvaluator(labelCol="DEP_DEL15", metricName="f1")
eval_auc = BinaryClassificationEvaluator(labelCol="DEP_DEL15", metricName="areaUnderROC")

# Calcular métricas
metricas = {
    "Árbol de Decisión": {
        "Accuracy": eval_acc.evaluate(pred_arbol),
        "F1-Score": eval_f1.evaluate(pred_arbol),
        "AUC-ROC": eval_auc.evaluate(pred_arbol)
    },
    "Random Forest": {
        "Accuracy": eval_acc.evaluate(pred_bosque),
        "F1-Score": eval_f1.evaluate(pred_bosque),
        "AUC-ROC": eval_auc.evaluate(pred_bosque)
    }
}

print("\n" + "="*60)
print("📊 COMPARACIÓN DE MÉTRICAS")
print("="*60)
print(f"{'Métrica':<15} {'Árbol':>15} {'Random Forest':>15} {'Ganador':>12}")
print("-"*60)

for metrica in ["Accuracy", "F1-Score", "AUC-ROC"]:
    val_arbol = metricas["Árbol de Decisión"][metrica]
    val_bosque = metricas["Random Forest"][metrica]
    ganador = "🌲 Bosque" if val_bosque > val_arbol else "🌳 Árbol"
    print(f"{metrica:<15} {val_arbol:>14.4f} {val_bosque:>15.4f} {ganador:>12}")

print("="*60)


📊 COMPARACIÓN DE MÉTRICAS
Métrica                   Árbol   Random Forest      Ganador
------------------------------------------------------------
Accuracy                0.5791          0.5842     🌲 Bosque
F1-Score                0.6254          0.6300     🌲 Bosque
AUC-ROC                 0.5890          0.6688     🌲 Bosque


## 4.4 Matriz de Confusión

In [4]:
from pyspark.sql.functions import col, count, when

def matriz_confusion(predicciones, nombre):
    print(f"\n📊 MATRIZ DE CONFUSIÓN - {nombre}")
    print("="*50)
    
    # Calcular componentes
    tp = predicciones.filter((col("DEP_DEL15") == 1) & (col("prediction") == 1)).count()
    tn = predicciones.filter((col("DEP_DEL15") == 0) & (col("prediction") == 0)).count()
    fp = predicciones.filter((col("DEP_DEL15") == 0) & (col("prediction") == 1)).count()
    fn = predicciones.filter((col("DEP_DEL15") == 1) & (col("prediction") == 0)).count()
    
    print(f"\n                    PREDICCIÓN")
    print(f"                 Puntual  Retrasado")
    print(f"         Puntual   {tn:>7,}   {fp:>7,}")
    print(f"REAL")
    print(f"       Retrasado   {fn:>7,}   {tp:>7,}")
    
    print(f"\n📌 Interpretación:")
    print(f"   ✅ True Positives (TP):  {tp:>10,} - Retrasos correctamente predichos")
    print(f"   ✅ True Negatives (TN):  {tn:>10,} - Puntuales correctamente predichos")
    print(f"   ❌ False Positives (FP): {fp:>10,} - Falsas alarmas")
    print(f"   ❌ False Negatives (FN): {fn:>10,} - Retrasos NO detectados")
    
    # Métricas derivadas
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    print(f"\n📈 Métricas derivadas:")
    print(f"   Precision: {precision:.4f} - De los que predije retraso, ¿cuántos acerté?")
    print(f"   Recall:    {recall:.4f} - De los retrasos reales, ¿cuántos detecté?")
    
    return {"TP": tp, "TN": tn, "FP": fp, "FN": fn}

cm_arbol = matriz_confusion(pred_arbol, "ÁRBOL DE DECISIÓN")
cm_bosque = matriz_confusion(pred_bosque, "RANDOM FOREST")


📊 MATRIZ DE CONFUSIÓN - ÁRBOL DE DECISIÓN

                    PREDICCIÓN
                 Puntual  Retrasado
         Puntual   482,927   384,504
REAL
       Retrasado    64,170   134,268

📌 Interpretación:
   ✅ True Positives (TP):     134,268 - Retrasos correctamente predichos
   ✅ True Negatives (TN):     482,927 - Puntuales correctamente predichos
   ❌ False Positives (FP):    384,504 - Falsas alarmas
   ❌ False Negatives (FN):     64,170 - Retrasos NO detectados

📈 Métricas derivadas:
   Precision: 0.2588 - De los que predije retraso, ¿cuántos acerté?
   Recall:    0.6766 - De los retrasos reales, ¿cuántos detecté?

📊 MATRIZ DE CONFUSIÓN - RANDOM FOREST

                    PREDICCIÓN
                 Puntual  Retrasado
         Puntual   486,303   381,128
REAL
       Retrasado    62,082   136,356

📌 Interpretación:
   ✅ True Positives (TP):     136,356 - Retrasos correctamente predichos
   ✅ True Negatives (TN):     486,303 - Puntuales correctamente predichos
   ❌ False Positiv

## 4.5 Análisis de Costos (para Finanzas)

In [5]:
# Costo estimado por tipo de error
COSTO_FN = 150  # Costo de un retraso NO predicho (compensación)
AHORRO_TP = 100  # Ahorro por predecir correctamente un retraso

print("\n💰 ANÁLISIS FINANCIERO")
print("="*55)
print(f"Supuestos: Costo por FN = ${COSTO_FN} | Ahorro por TP = ${AHORRO_TP}")
print("-"*55)

for nombre, cm in [("Árbol", cm_arbol), ("Bosque", cm_bosque)]:
    ahorro = cm["TP"] * AHORRO_TP
    perdida = cm["FN"] * COSTO_FN
    balance = ahorro - perdida
    
    print(f"\n{nombre}:")
    print(f"  💚 Ahorro (TP × ${AHORRO_TP}):  ${ahorro:>12,}")
    print(f"  💔 Pérdida (FN × ${COSTO_FN}): ${perdida:>12,}")
    print(f"  {'📈' if balance > 0 else '📉'} Balance neto:       ${balance:>12,}")


💰 ANÁLISIS FINANCIERO
Supuestos: Costo por FN = $150 | Ahorro por TP = $100
-------------------------------------------------------

Árbol:
  💚 Ahorro (TP × $100):  $  13,426,800
  💔 Pérdida (FN × $150): $   9,625,500
  📈 Balance neto:       $   3,801,300

Bosque:
  💚 Ahorro (TP × $100):  $  13,635,600
  💔 Pérdida (FN × $150): $   9,312,300
  📈 Balance neto:       $   4,323,300


## 4.6 Reglas del Árbol (para Marketing)

In [6]:
# Extraer reglas interpretables del árbol
print("\n📢 REGLAS PARA SISTEMA DE ALERTAS")
print("="*50)
print("\nBasado en el árbol de decisión, estas son las condiciones")
print("que más aumentan la probabilidad de retraso:\n")

# Mostrar estructura simplificada
reglas = modelo_arbol.toDebugString
lineas = reglas.split("\n")[:30]
for linea in lineas:
    print(linea)


📢 REGLAS PARA SISTEMA DE ALERTAS

Basado en el árbol de decisión, estas son las condiciones
que más aumentan la probabilidad de retraso:

DecisionTreeClassificationModel: uid=DecisionTreeClassifier_17a58d9b25b8, depth=8, numNodes=233, numClasses=2, numFeatures=7
  If (feature 2 <= 10.5)
   If (feature 2 <= 7.5)
    If (feature 5 in {0.0,3.0,5.0,6.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,29.0,30.0,31.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,64.0,65.0,66.0,67.0,68.0,69.0,70.0,71.0,72.0,73.0,74.0,75.0,76.0,77.0,78.0,79.0,80.0,81.0,82.0,83.0,84.0,85.0,86.0,87.0,88.0,89.0,90.0,91.0,92.0,93.0,94.0,95.0,96.0,97.0,98.0,99.0,100.0,101.0,102.0,103.0,104.0,105.0,106.0,107.0,108.0,109.0,110.0,111.0,112.0,113.0,114.0,115.0,116.0,117.0,118.0,119.0,120.0,121.0,122.0,123.0,124.0,125.0,126.0,127.0,128.0,129.0,130.0,131.

## 4.7 Tabla Comparativa Final

In [7]:
print("\n" + "="*70)
print("🏆 TABLA COMPARATIVA FINAL")
print("="*70)
print(f"{'Criterio':<25} {'Árbol':>20} {'Random Forest':>20}")
print("-"*70)
print(f"{'Accuracy':<25} {metricas['Árbol de Decisión']['Accuracy']:>19.2%} {metricas['Random Forest']['Accuracy']:>19.2%}")
print(f"{'F1-Score':<25} {metricas['Árbol de Decisión']['F1-Score']:>19.4f} {metricas['Random Forest']['F1-Score']:>19.4f}")
print(f"{'AUC-ROC':<25} {metricas['Árbol de Decisión']['AUC-ROC']:>19.4f} {metricas['Random Forest']['AUC-ROC']:>19.4f}")
print(f"{'Interpretabilidad':<25} {'✅ Alta':>20} {'❌ Baja':>20}")
print(f"{'Velocidad':<25} {'✅ Rápido':>20} {'⚠️ Lento':>20}")
print(f"{'Producción (tiempo real)':<25} {'✅ Recomendado':>20} {'⚠️ Evaluar':>20}")
print("="*70)


🏆 TABLA COMPARATIVA FINAL
Criterio                                 Árbol        Random Forest
----------------------------------------------------------------------
Accuracy                               57.91%              58.42%
F1-Score                               0.6254              0.6300
AUC-ROC                                0.5890              0.6688
Interpretabilidad                       ✅ Alta               ❌ Baja
Velocidad                             ✅ Rápido             ⚠️ Lento
Producción (tiempo real)         ✅ Recomendado           ⚠️ Evaluar


---
## ✅ CHECKPOINT MÓDULO 4 - FINAL

Ahora puedes responder las preguntas de TU MISIÓN usando:

- Métricas de evaluación
- Matriz de confusión
- Feature importance
- Reglas del árbol
- Análisis de costos

**¡Prepara tu recomendación para el CEO!** 🎯

In [8]:
# Función para recuperar las etiquetas
def recuperar_etiquetas(df, nombre_columna_idx):
    meta = df.schema[nombre_columna_idx].metadata
    return meta['ml_attr']['vals']

dicc_airline = recuperar_etiquetas(df, "AIRLINE_idx")
dicc_origin  = recuperar_etiquetas(df, "ORIGIN_idx")
dicc_dest    = recuperar_etiquetas(df, "DEST_idx")

# Imprimir TODAS las aerolíneas
print("✈️ TODAS LAS AEROLÍNEAS:")
for i, nombre in enumerate(dicc_airline):
    print(f"{float(i)}: {nombre}")

# Imprimir TODOS los orígenes (¡Aquí puede haber más de 300!)
print("\n🏙️ TODOS LOS ORÍGENES:")
for i, nombre in enumerate(dicc_origin):
    print(f"{float(i)}: {nombre}")

✈️ TODAS LAS AEROLÍNEAS:
0.0: WN
1.0: DL
2.0: AA
3.0: OO
4.0: EV
5.0: UA
6.0: MQ
7.0: B6
8.0: US
9.0: AS
10.0: NK
11.0: F9
12.0: HA
13.0: VX
14.0: __unknown

🏙️ TODOS LOS ORÍGENES:
0.0: ATL
1.0: ORD
2.0: DFW
3.0: DEN
4.0: LAX
5.0: SFO
6.0: PHX
7.0: IAH
8.0: LAS
9.0: MSP
10.0: MCO
11.0: SEA
12.0: DTW
13.0: BOS
14.0: EWR
15.0: CLT
16.0: LGA
17.0: SLC
18.0: JFK
19.0: BWI
20.0: MDW
21.0: DCA
22.0: FLL
23.0: SAN
24.0: MIA
25.0: PHL
26.0: TPA
27.0: DAL
28.0: HOU
29.0: BNA
30.0: PDX
31.0: STL
32.0: HNL
33.0: OAK
34.0: AUS
35.0: MSY
36.0: MCI
37.0: SJC
38.0: SMF
39.0: SNA
40.0: CLE
41.0: IAD
42.0: RDU
43.0: MKE
44.0: SAT
45.0: RSW
46.0: IND
47.0: SJU
48.0: CMH
49.0: PIT
50.0: PBI
51.0: OGG
52.0: CVG
53.0: ABQ
54.0: BUR
55.0: BDL
56.0: JAX
57.0: ONT
58.0: BUF
59.0: OMA
60.0: OKC
61.0: ANC
62.0: RIC
63.0: TUS
64.0: MEM
65.0: TUL
66.0: RNO
67.0: BHM
68.0: ELP
69.0: CHS
70.0: BOI
71.0: KOA
72.0: PVD
73.0: GRR
74.0: LIH
75.0: LIT
76.0: SDF
77.0: GEG
78.0: ORF
79.0: XNA
80.0: MSN
81.0: PSP
82.0: LGB

In [16]:
# 1. Definición de la "Intemperie" (Escenario Base)
# Todos los retrasos reales evaluados al costo total de compensación ($150)
total_retrasos_reales = cm_bosque["TP"] + cm_bosque["FN"]
costo_intemperie = total_retrasos_reales * 150

# 2. Configuración de escenarios
COSTO_FN = 150  
ESCENARIOS_AHORRO = [100,150]

print("💰 REPORTE DE IMPACTO FINANCIERO - COMPARATIVA DE ESCENARIOS")
print("="*75)
print(f"VALOR DEL PROBLEMA (Costo total sin modelo): ${costo_intemperie:>14,.0f}")
print("-" * 75)

for ahorro_unid in ESCENARIOS_AHORRO:
    header = f"Escenario: Ahorro de ${ahorro_unid} por TP"
    print(f"\n🔹 {header}")
    print("-" * 45)
    
    for nombre, cm in [("Árbol", cm_arbol), ("Bosque", cm_bosque)]:
        ahorro_logrado = cm["TP"] * ahorro_unid
        perdida_por_fallos = cm["FN"] * COSTO_FN
        balance_neto = ahorro_logrado - perdida_por_fallos
        perdida_por_fallos = cm["FN"] * COSTO_FN
        
        # Eficiencia basada en el ahorro bruto capturado vs costo total
        eficiencia = (ahorro_logrado / costo_intemperie) * 100
        
        print(f"{nombre:>7}: Ahorro: ${ahorro_logrado:>11,.0f} | gasto no evitado: ${perdida_por_fallos:>11,.0f} | Balance: ${balance_neto:>11,.0f} | Eficiencia: {eficiencia:>6.2f}%")

print("\n" + "="*75)

💰 REPORTE DE IMPACTO FINANCIERO - COMPARATIVA DE ESCENARIOS
VALOR DEL PROBLEMA (Costo total sin modelo): $    29,765,700
---------------------------------------------------------------------------

🔹 Escenario: Ahorro de $100 por TP
---------------------------------------------
  Árbol: Ahorro: $ 13,426,800 | gasto no evitado: $  9,625,500 | Balance: $  3,801,300 | Eficiencia:  45.11%
 Bosque: Ahorro: $ 13,635,600 | gasto no evitado: $  9,312,300 | Balance: $  4,323,300 | Eficiencia:  45.81%

🔹 Escenario: Ahorro de $150 por TP
---------------------------------------------
  Árbol: Ahorro: $ 20,140,200 | gasto no evitado: $  9,625,500 | Balance: $ 10,514,700 | Eficiencia:  67.66%
 Bosque: Ahorro: $ 20,453,400 | gasto no evitado: $  9,312,300 | Balance: $ 11,141,100 | Eficiencia:  68.71%

